<a href="https://colab.research.google.com/github/Chosencodes/Lung-Segmentation/blob/main/Lung_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
torch.set_float32_matmul_precision("high")

In [2]:
# !wget -q --show-progress https://msd-for-monai.s3-us-west-2.amazonaws.com/Task06_Lung.tar -O /content/Task06_Lung.tar
# !tar -xf /content/Task06_Lung.tar -C /content/
# !ls /content/Task06_Lung        # -> imagesTr  labelsTr  dataset.json

In [3]:
# from google.colab import drive
# drive.mount('/content/drive')

# !mkdir -p "/content/drive/MyDrive/datasets"

# !cp /content/Task06_Lung.tar "/content/drive/MyDrive/datasets/Task06_Lung.tar"
# !ls -lh "/content/drive/MyDrive/datasets/"

In [13]:
!ls -lh "/content/drive/MyDrive/datasets"


total 8.6G
-rw------- 1 root root 8.6G Jul  7 11:26 Task06_Lung.tar


In [14]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!cp "/content/drive/MyDrive/datasets/Task06_Lung.tar" /content/

In [ ]:
!ls -lh /content/Task06_Lung.tar

In [ ]:
!tar -xf /content/Task06_Lung.tar -C /content/

In [4]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r '/content/drive/MyDrive/Task06_Lung' '/content/Task06_Lung'

data_path = '/content/Task06_Lung'

Mounted at /content/drive
cp: cannot stat '/content/drive/MyDrive/Task06_Lung': No such file or directory


In [11]:
import os

for item in os.listdir("/content/drive/MyDrive"):
    print(item)

Getting started.pdf
Edugain Application form .docx
unknown
rsna_heart_detection.csv
C6D5F1B9-C69E-4FA0-B7FF-C84240F196F1.jpeg
Resumee (2).gdoc
Cover letter.docx
Unicef.gdoc
Unicef.docx
Seminar.gdoc
Resumee (1).docx
Resumee.docx
Transcript.gdoc
Resume (4).gdoc
Resume (3).gdoc
Resume.docx
Project.gdoc
Copy of Kid's Church Leader Guidelines & Volunteer Contract.gdoc
Copy of Kids Church Interview.gdoc
Copy of Kids Church Job Description.gdoc
Children Presentation.gdoc
IMG_3067 (2).MOV
IMG_3067 (1).MOV
IMG_3067.MOV
Resumee (1).gdoc
Copy of Week 1 - Welcome Video - Professional Foundations.gdoc
Chosen_Professional Foundations_Folder
Copy of Professional Foundations - Week 1 Milestone Rubric (1).gdoc
Copy of ALX Professional Foundations - Skills Tracker (4).gsheet
Copy of ALX Professional Foundations - Skills Tracker (3).gsheet
Copy of ALX Professional Foundations - Skills Tracker (2).gsheet
Copy of ALX Professional Foundations - Skills Tracker (1).gsheet
Copy of PICS and Personal Mission Sta

In [5]:
#  !ls -la /content/Task06_Lung

In [6]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import os

In [7]:
root = Path("/content/Task06_Lung/imagesTr")
label = Path("/content/Task06_Lung/labelsTr")

In [8]:
sample_path = next(root.glob("lung*.nii.gz"))
sample_label_path = label / sample_path.name

StopIteration: 

In [ ]:
data = nib.load(sample_path)
label = nib.load(sample_label_path)

In [ ]:
ct_scan = data.get_fdata()
mask = label.get_fdata().astype(np.uint8)

In [ ]:
print(data.shape)
print(label.shape)

print(nib.aff2axcodes(data.affine))
print(nib.aff2axcodes(label.affine))
print(data.header.get_zooms())
print(np.unique(mask))

In [ ]:
tumor_pixels = np.sum(mask == 1)
background_pixels = np.sum(mask == 0)

print(tumor_pixels)
print(background_pixels)

print("Tumor %:", tumor_pixels / mask.size * 100)

In [ ]:
case = "lung_054.nii.gz"
print("Patient:", case)

image = nib.load(f"/content/Task06_Lung/imagesTr/{case}").get_fdata()
label = nib.load(f"/content/Task06_Lung/labelsTr/{case}").get_fdata().astype(np.uint8)

tumor_slices = np.where(label.sum(axis=(0, 1)) > 0)[0]

print("Tumor slices:", tumor_slices)

z = tumor_slices[np.argmax(label.sum(axis=(0, 1))[tumor_slices])]

plt.figure(figsize=(7,7))

plt.imshow(np.clip(image[:, :, z], -1000, 400).T,
           cmap="gray",
           origin="lower")

plt.imshow(np.ma.masked_where(label[:, :, z] == 0,
                              label[:, :, z]).T,
           cmap="autumn",
           alpha=0.6,
           origin="lower")

plt.title(f"{case} - Slice {z}")
plt.axis("off")
plt.show()

# **Preprocessing**

In [ ]:
!pip install torchio nibabel scikit-learn tqdm -q

In [ ]:
import torchio as tio
from sklearn.model_selection import train_test_split,KFold
from torch.utils.data import DataLoader

In [ ]:
image_path = sorted(p for p in root.glob("lung*.nii.gz")  if not p.name.startswith("."))
label_path = sorted(p for p in label.glob("lung*.nii.gz") if not p.name.startswith("."))
print("images:", len(image_path), "labels:", len(label_path))

In [ ]:
def make_subject(image_path,label_path):
  subjects = []
  for img_path,lbl_path in zip(image_path,label_path):
    subject = tio.Subject(
        mri = tio.ScalarImage(img_path),
        mask = tio.LabelMap(lbl_path)
    )
    subjects.append(subject)
  return subjects

all_subject = make_subject(image_path,label_path)

In [ ]:
base_transform = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1.5),
    tio.Clamp(-1000,400),
    tio.RescaleIntensity(
        out_min_max=(0, 1),
        in_min_max=(-1000, 400),
    ),
])

In [ ]:
train_transform = tio.Compose([
    base_transform,
    tio.RandomFlip(axes=(0,), flip_probability=0.5),
    tio.RandomAffine(scales=(0.9, 1.1), degrees=10),
    tio.RandomNoise(std=(0, 0.03)),
])

val_transform = base_transform

In [ ]:
kf = KFold(n_splits=3, shuffle=True, random_state=42)
PATCH_SIZE = (96, 96, 96)

sampler = tio.LabelSampler(
    patch_size=PATCH_SIZE,
    label_name="mask",
    label_probabilities={0: 0.2, 1: 0.8},
)

def build_fold_loaders(train_subjects, val_subjects):
    train_ds = tio.SubjectsDataset(train_subjects, transform=train_transform)
    train_queue = tio.Queue(
        subjects_dataset=train_ds,
        max_length=64,
        samples_per_volume=8,
        sampler=sampler,
        num_workers=4,
        shuffle_subjects=True,
        shuffle_patches=True,
    )
    train_loader = tio.SubjectsLoader(train_queue, batch_size=2)

    val_ds = tio.SubjectsDataset(val_subjects, transform=val_transform)
    val_loader = tio.SubjectsLoader(val_ds, batch_size=1, num_workers=2, shuffle=False)

    return train_loader, val_loader

print("Preprocessing setup complete.")

# **Model**

In [ ]:
!pip install monai pytorch-lightning -q

In [ ]:
import pytorch_lightning as pl
from monai.networks.nets import UNet
from monai.losses import DiceCELoss,DiceFocalLoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger

In [ ]:
def build_model():
  return UNet(
      spatial_dims = 3,
      in_channels = 1,
      out_channels = 1,
      channels=(16,32,64,128,256),
      strides=(2,2,2,2),
      num_res_units=2,
      dropout=0.1,
  )

In [ ]:
class LungTumorSegmentation(pl.LightningModule):
    def __init__(self, lr=2e-4):
        super().__init__()
        self.save_hyperparameters()

        self.model = build_model()
        self.loss_fn = DiceFocalLoss(sigmoid=True, gamma=2.0,lambda_dice=1.0, lambda_focal=1.0)
        self.metrics = DiceMetric(include_background=False, reduction="mean")
        self.patch_size = (96, 96, 96)

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        mri  = batch["mri"]["data"]
        mask = batch["mask"]["data"].float()
        pred = self.model(mri)
        loss = self.loss_fn(pred, mask)
        self.log("train_loss", loss, prog_bar=True, batch_size=mri.shape[0])
        return loss

    def validation_step(self, batch, batch_idx):
        mri  = batch["mri"]["data"]
        mask = batch["mask"]["data"].float()
        pred = sliding_window_inference(
            inputs=mri, roi_size=self.patch_size, sw_batch_size=4,
            predictor=self.forward, overlap=0.5, mode="gaussian",
        )
        loss = self.loss_fn(pred, mask)
        pred_bin = (torch.sigmoid(pred) > 0.5).float()
        self.metrics(pred_bin, mask)
        self.log("val_loss", loss, prog_bar=True, batch_size=1)
        if batch_idx == 0:
            self.log_images(mri.cpu(), pred.cpu(), mask.cpu(), "val")
        return loss

    def on_validation_epoch_end(self):
        dice = self.metrics.aggregate().item()
        self.metrics.reset()
        self.log("val_dice", dice, prog_bar=True)
        print(f"Val dice: {dice:.4f}")

    def log_images(self, mri, logits, mask, name):
        d = mri.shape[-1] // 2
        pred_bin = (torch.sigmoid(pred) > 0.5).float()
        ct = mri[0, 0, :, :, d]
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        for ax, m, title in [(axes[0], mask, "Ground Truth"),
                             (axes[1], pred_bin, "Prediction")]:
            ax.imshow(ct, cmap="gray")
            ov = np.ma.masked_where(m[0, 0, :, :, d] == 0, m[0, 0, :, :, d])
            ax.imshow(ov, alpha=0.6, cmap="autumn")
            ax.set_title(title); ax.axis("off")
        plt.suptitle(f"{name} — slice {d}")
        plt.tight_layout()
        self.logger.experiment.add_figure(name, fig, self.global_step)
        plt.close(fig)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.hparams.lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.5, patience=6)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "monitor": "val_dice"},
        }

In [ ]:
import shutil
shutil.rmtree("/content/drive/MyDrive/lung_checkpoints", ignore_errors=True)

In [ ]:
pl.seed_everything(42, workers=True)
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(all_subject)):
    fold_dir    = f"/content/drive/MyDrive/lung_checkpoints/fold_{fold+1}"
    done_marker = os.path.join(fold_dir, "done.txt")
    os.makedirs(fold_dir, exist_ok=True)

    if os.path.exists(done_marker):
        with open(done_marker) as f:
            fold_scores.append(float(f.read().strip().split("=")[1]))
        print(f"Fold {fold+1} already done ({fold_scores[-1]:.4f}) — skipping")
        continue

    print(f"\n===== FOLD {fold+1}/{kf.get_n_splits()} =====")
    train_subjects = [all_subject[i] for i in train_idx]
    val_subjects   = [all_subject[i] for i in val_idx]
    train_loader, val_loader = build_fold_loaders(train_subjects, val_subjects)

    model = LungTumorSegmentation(lr=3e-4)
    checkpoint = ModelCheckpoint(
        monitor="val_dice", mode="max", save_top_k=1, save_last=True,
        dirpath=fold_dir, filename="best-{epoch}-{val_dice:.3f}",
    )
    early_stop = EarlyStopping(monitor="val_dice", mode="max", patience=25)
    lr_mon = LearningRateMonitor(logging_interval="epoch")
    trainer = pl.Trainer(
          accelerator="gpu", devices=1,
          max_epochs=120, min_epochs=40,
          precision="16-mixed", gradient_clip_val=1.0,
          callbacks=[checkpoint, early_stop, lr_mon],
          logger=TensorBoardLogger(save_dir="./logs", name=f"lung_fold{fold+1}"),
          log_every_n_steps=5, num_sanity_val_steps=0,
)
    last_ckpt = os.path.join(fold_dir, "last.ckpt")
    resume = last_ckpt if os.path.exists(last_ckpt) else None
    trainer.fit(model, train_loader, val_loader, ckpt_path=resume)

    best = checkpoint.best_model_score.item()
    fold_scores.append(best)
    with open(done_marker, "w") as f:
        f.write(f"best_dice={best:.4f}")
    print(f"Fold {fold+1} best Dice: {best:.4f}")

print(f"\nPer-fold: {[round(s, 4) for s in fold_scores]}")
print(f"Mean Dice: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

In [ ]:
import os, numpy as np

scores = []
for fold in range(1, 4):
    marker = f"/content/drive/MyDrive/lung_checkpoints/fold_{fold}/done.txt"
    if os.path.exists(marker):
        with open(marker) as f:
            content = f.read().strip()          # e.g. "best_dice=0.3228"
            dice = float(content.split("=")[1])
            scores.append(dice)
            print(f"Fold {fold}: {dice:.4f}")
    else:
        print(f"Fold {fold}: not finished (no done.txt)")

if scores:
    print(f"\nMean Dice: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

In [ ]:
import os
base = "/content/drive/MyDrive/lung_checkpoints"
for fold in [2, 3]:
    d = f"{base}/fold_{fold}"
    print(f"fold_{fold}:", os.listdir(d))